# Extract From Bigquery

### Imports

In [1]:
import pandas as pd                                   # tables
from pathlib import Path                              # file paths that work on every operating system
from google.cloud import bigquery                     # official BigQuery client for Python
import pydata_google_auth                             # opens a browser so you can sign in with your Google account

### Settings and paths

In [2]:
PROJECT_ID = "flood-it-retention"                        # REPLACE with your Google Cloud project ID (shown in the BigQuery console)
ROOT = Path.cwd()                                     # folder the notebook runs in
if ROOT.name == "notebooks":                          # if we are inside notebooks/...
    ROOT = ROOT.parent                                # ...go up to the project root
SQL_DIR = ROOT / "sql"                                # where the .sql files live
OUT_DIR = ROOT / "data" / "processed"                 # where exported tables will be saved
OUT_DIR.mkdir(parents=True, exist_ok=True)            # create the folder if it does not exist

In [3]:
SCOPES = ["https://www.googleapis.com/auth/cloud-platform"]          # permission to use BigQuery on your behalf
credentials = pydata_google_auth.get_user_credentials(SCOPES)        # browser sign-in, then cached on your computer
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)  # the connection every query will use
print(client.query("SELECT 'connected' AS status").to_dataframe())  # quick test: should print one row saying connected

      status
0  connected


### Exploration Queries

In [4]:
explore_files = sorted(SQL_DIR.glob("0[1-5]_*.sql"))                 # files 01 to 05
for sql_file in explore_files:                                       # loop over them one by one
    result = client.query(sql_file.read_text()).to_dataframe()      # run the query and download the result
    result.to_csv(OUT_DIR / f"explore_{sql_file.stem}.csv", index=False)  # save as CSV
    print(sql_file.name, result.shape)                               # confirm rows and columns
    print(result.head(10).to_string(index=False))                    # show the first rows

01_overview.sql (1, 5)
 first_day   last_day  days_of_data  players  events
2018-06-12 2018-10-03           114    15175 5700000
02_event_counts.sql (37, 4)
              event_name  events  players  share_of_events_pct
             screen_view 2247623    14077                39.43
         user_engagement 1358958    13588                23.84
   level_start_quickplay  523430    10166                 9.18
     level_end_quickplay  349729     8168                 6.14
              post_score  242051     8580                 4.25
level_complete_quickplay  191088     5676                 3.35
    level_fail_quickplay  137035     6343                 2.40
   level_reset_quickplay  122278     3517                 2.15
          select_content  105139    11111                 1.84
             level_start   74417     4774                 1.31
03_event_param_keys.sql (205, 6)
    event_name             param_key  occurrences  text_values  integer_values  decimal_values
     ad_reward firebas

### Creating dataset

In [5]:
build_files = [SQL_DIR / "00_create_dataset.sql"] + sorted(SQL_DIR.glob("0[6-9]_*.sql")) + sorted(SQL_DIR.glob("1[0-7]_*.sql"))  # build order matters
for sql_file in build_files:                                         # loop over the files in order
    job = client.query(sql_file.read_text())                         # send the CREATE statement to BigQuery
    job.result()                                                     # wait until BigQuery finishes
    print("created:", sql_file.name)                                 # confirm

created: 00_create_dataset.sql
created: 06_create_stg_events.sql
created: 07_create_dim_players.sql
created: 08_create_fct_player_days.sql
created: 09_create_fct_sessions.sql
created: 10_create_kpi_daily.sql
created: 11_create_player_retention.sql
created: 12_create_retention_summary.sql
created: 13_create_retention_cohorts_weekly.sql
created: 14_create_player_features.sql
created: 15_create_retention_by_milestone.sql
created: 16_create_first_day_funnel.sql
created: 17_create_retention_by_segment.sql


### Export to CSV

In [6]:
views = ["kpi_daily", "retention_summary", "retention_cohorts_weekly", "retention_by_milestone",
         "first_day_funnel", "retention_by_segment", "player_features"]          # tables the later notebooks read
for view in views:                                                               # loop over each view
    table = client.query(f"SELECT * FROM `{PROJECT_ID}.flood_it.{view}`").to_dataframe()  # download the whole view
    table.to_csv(OUT_DIR / f"{view}.csv", index=False)                           # save it
    print(view, table.shape)                                                     # rows and columns

kpi_daily (114, 13)
retention_summary (5, 5)
retention_cohorts_weekly (426, 5)
retention_by_milestone (4, 6)
first_day_funnel (6, 3)
retention_by_segment (22, 6)
player_features (11066, 28)
